# Name : Bansi Ajagia
# **Subject** : Machine Learning
# **Project** : Cardiovascular Disease Prediction System

---

# Task 5 — Advanced Model Training

## 📥 Prepare Data

In [1]:
# Name : Bansi Ajagia
# Subject : Machine Learning
# Project : Cardiovascular Disease Prediction System

import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

df = pd.read_csv('../data/cardio.csv', sep=';')
df['age_years'] = (df['age'] / 365).round(0).astype(int)
df = df[df['ap_hi'].between(60,250)]
df = df[df['ap_lo'].between(40,200)]
df = df[df['height'].between(100,250)]
df = df[df['weight'].between(30,250)]

FEATURE_COLS = ['age_years','gender','height','weight',
                'ap_hi','ap_lo','cholesterol','gluc','smoke','alco','active']
X = df[FEATURE_COLS].values
y = df['cardio'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Data ready.')

Data ready.


## 1️⃣ Cross-Validation (5-Fold)

In [3]:
lr = LogisticRegression(max_iter=1000, random_state=42)
cv_scores = cross_val_score(lr, X_train_sc, y_train, cv=5, scoring='accuracy')
print('Logistic Regression 5-Fold CV:')
print(f'  Scores : {cv_scores.round(4)}')
print(f'  Mean   : {cv_scores.mean():.4f}')
print(f'  Std    : {cv_scores.std():.4f}')

Logistic Regression 5-Fold CV:
  Scores : [0.7333 0.7285 0.7254 0.7243 0.7312]
  Mean   : 0.7286
  Std    : 0.0034


## 2️⃣ Hyperparameter Tuning — GridSearchCV

In [5]:
# Keeping small grid for speed
param_grid = {
    'n_estimators': [50, 100],
    'max_depth'   : [5, 10, None],
    'min_samples_split': [2, 5],
}
rf = RandomForestClassifier(random_state=42)
grid = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

# Use a subset for speed (10 000 rows)
from sklearn.utils import resample
X_sub, y_sub = resample(X_train, y_train, n_samples=10000, random_state=42)
grid.fit(X_sub, y_sub)

print(f'Best params: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best CV accuracy: 0.7464


## 3️⃣ Final Comparison with Best RF

In [8]:
best_rf = grid.best_estimator_
best_rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, best_rf.predict(X_test))

lr.fit(X_train_sc, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test_sc))

print(f'Logistic Regression (tuned): {lr_acc:.4f}')
print(f'Random Forest (tuned): {rf_acc:.4f}')

Logistic Regression (tuned): 0.7224
Random Forest (tuned): 0.7119
